# Fase 2 — Transformação dos Dados no Dataset
---

## Objetivo 

Identificar colunas que precisam de alterações

- Resolver colunas numéricas divididas em intervalos
- Tratar colunas com valores nulos em classifições
- Partir de colunas categóricas para valores numéricos
- Resolver colunas que são valores compostos (listas)
- Usar a coluna do CNAE para melhor identificação
- Criar colunas derivadas a partir de dados dos clientes


---

## Datasets disponíveis

| Arquivo | Descrição |
|---|---|
| `credito_aplicacao_clientes.csv` | Dados cadastrais de 200 clientes (Serasa, iFood, Google Maps, etc.) |
| `credito_comportamental_pedidos.csv` | Histórico de 200 pedidos de clientes já ativos |

## 0. Importações

- `pandas`: manipulação e análise de dados tabulares
- `numpy`: operações numéricas
- `os`: Escrever arquivo csv a partir do df

In [41]:
import pandas as pd
import numpy as np
import os
print("Importações Concluídas")

Importações Concluídas


## 1. Carregando os Dados

In [28]:
# Carregamento dos dois datasets
df_clientes = pd.read_csv('../dados/credito_aplicacao_clientes.csv', sep=',')
df_pedidos  = pd.read_csv('../dados/credito_comportamental_pedidos.csv', sep=',')

print('Datasets carregados')
print(f'  Clientes: {df_clientes.shape[0]} linhas x {df_clientes.shape[1]} colunas')
print(f'  Pedidos:  {df_pedidos.shape[0]} linhas x {df_pedidos.shape[1]} colunas')

Datasets carregados
  Clientes: 200 linhas x 20 colunas
  Pedidos:  200 linhas x 5 colunas


In [29]:
df_clientes.head()

,id_cliente,uf,municipio,segmento_cliente,natureza_juridica,fonte_cliente,cnae_codigo,cnae_descricao,capital_social,idade_cnpj,serasa_contagem_negativacoes,serasa_contagem_protestos,serasa_credores,serasa_socio_tem_negativacao,ifood_contagem_avaliacoes,ifood_faixa_preco,google_maps_avaliacao,google_maps_contagem_avaliacoes,google_maps_tem_website,inadimplente
0,5649308221613,PE,RECIFE,Restaurante Brasileiro,230-5 - Empresa Individual de Responsabilidade...,Fonte 5,56.11-2-01,Restaurantes e similares,"(100000, 200000]","(8000, 25000]",0.0,0.0,NaN,0.0,"(200, 500]",$$$$$,"(4.0, 4.5]","(1000, 2500]",1.0,0.0
1,5728626180269,PE,RECIFE,Padaria,230-5 - Empresa Individual de Responsabilidade...,Fonte 5,10.91-1-02,Fabricação de produtos de padaria e confeitari...,"(70000, 100000]","(8000, 25000]",0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
2,5740206260397,PE,JABOATAO DOS GUARARAPES,Conveniência,206-2 - Sociedade Empresária Limitada,Fonte 5,46.13-3-00,Representantes comerciais e agentes do comérci...,"(8000, 10000]","(5000, 8000]",0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
3,5741028343981,PE,FERNANDO DE NORONHA,Hotel,206-2 - Sociedade Empresária Limitada,Fonte 5,55.10-8-01,Hotéis,"(8000, 10000]","(5000, 8000]",0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
4,5741479985325,PE,RECIFE,Churrascaria,206-2 - Sociedade Empresária Limitada,Fonte 5,56.11-2-01,Restaurantes e similares,"(200000, 1000000]","(8000, 25000]",0.0,0.0,NaN,0.0,"(50, 100]",$,"(4.0, 4.5]","(30, 50]",1.0,0.0


In [30]:
df_clientes_tratado = df_clientes.copy() #criando uma cópia do dataframe original para tratamento
df_pedidos_tratado  = df_pedidos.copy()

## 2. Tratamento de Dados por Tipo

Esta seção é responsável pelo tratamento do dataset para uma versão sem os problemas de campos nulos, não numéricos e intervalos.

### 2.1 Resolução de colunas numéricas dividas em Intervalos


#### 2.1.1 Identificação de Intervalos

In [31]:
import re

# Padrão exato de intervalo numérico, ex.: (1000, 2500] ou [0, 5)
padrao_intervalo = re.compile(r"^\s*[\(\[]\s*-?\d+(?:[\.,]\d+)?\s*,\s*-?\d+(?:[\.,]\d+)?\s*[\)\]]\s*$")

colunas_intervalo = []

for coluna in df_clientes.columns:
    serie = df_clientes[coluna].dropna().astype(str).str.strip()
    if serie.empty:
        continue

    # Considera coluna de intervalo quando todos os valores não nulos seguem o padrão
    if serie.str.match(padrao_intervalo).all():
        colunas_intervalo.append(coluna)
        exemplos = serie.drop_duplicates().head(3).tolist()
        print(f"{coluna}: {exemplos}")

if not colunas_intervalo:
    print("Nenhuma coluna com padrão de intervalo foi encontrada.")
else:
    print("\nColunas identificadas:", colunas_intervalo)

capital_social: ['(100000, 200000]', '(70000, 100000]', '(8000, 10000]']
idade_cnpj: ['(8000, 25000]', '(5000, 8000]', '(4000, 5000]']
ifood_contagem_avaliacoes: ['(200, 500]', '(50, 100]', '(5, 10]']
google_maps_avaliacao: ['(4.0, 4.5]', '(4.5, 5.0]', '(3.0, 4.0]']
google_maps_contagem_avaliacoes: ['(1000, 2500]', '(30, 50]', '(200, 300]']

Colunas identificadas: ['capital_social', 'idade_cnpj', 'ifood_contagem_avaliacoes', 'google_maps_avaliacao', 'google_maps_contagem_avaliacoes']


#### 2.1.2 Transformação usando Mediana

Considerando este compormento e pelas análises feitas na **Fase 1**, será adotado como o uso da **mediana** como a a estratégia para transformar estas colunas de interlvalos para valores numéricos. Pois pela quantidade de dados do dataset, usar a média poderia trazer dificuldades para discrepâncias de valores.

In [32]:
# Conversao de intervalos para valor numerico usando a mediana da faixa (ponto medio)
intervalo_regex = re.compile(
    r"^\s*([\(\[])\s*(-?\d+(?:[\.,]\d+)?)\s*,\s*(-?\d+(?:[\.,]\d+)?)\s*([\)\]])\s*$"
 )

def intervalo_para_mediana(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip()
    match = intervalo_regex.match(texto)
    if not match:
        return np.nan

    limite_inferior = float(match.group(2).replace(',', '.'))
    limite_superior = float(match.group(3).replace(',', '.'))

    # Mediana da faixa para intervalo continuo: ponto medio
    return (limite_inferior + limite_superior) / 2.0


colunas_convertidas = []
for coluna in colunas_intervalo:
    serie_convertida = df_clientes_tratado[coluna].apply(intervalo_para_mediana)

    # So converte se toda a parte nao nula foi interpretada corretamente
    if serie_convertida.notna().sum() == df_clientes_tratado[coluna].notna().sum():
        df_clientes_tratado[coluna] = serie_convertida
        colunas_convertidas.append(coluna)

print('Colunas convertidas com mediana da faixa:')
print(colunas_convertidas)

if colunas_convertidas:
    print('\nAmostra apos conversao:')
    print(df_clientes_tratado[colunas_convertidas].head())

Colunas convertidas com mediana da faixa:
['capital_social', 'idade_cnpj', 'ifood_contagem_avaliacoes', 'google_maps_avaliacao', 'google_maps_contagem_avaliacoes']

Amostra apos conversao:
   capital_social  idade_cnpj  ifood_contagem_avaliacoes  \
0        150000.0     16500.0                      350.0   
1         85000.0     16500.0                        NaN   
2          9000.0      6500.0                        NaN   
3          9000.0      6500.0                        NaN   
4        600000.0     16500.0                       75.0   

   google_maps_avaliacao  google_maps_contagem_avaliacoes  
0                   4.25                           1750.0  
1                    NaN                              NaN  
2                    NaN                              NaN  
3                    NaN                              NaN  
4                   4.25                             40.0  


In [33]:
df_clientes_tratado.head()

,id_cliente,uf,municipio,segmento_cliente,natureza_juridica,fonte_cliente,cnae_codigo,cnae_descricao,capital_social,idade_cnpj,serasa_contagem_negativacoes,serasa_contagem_protestos,serasa_credores,serasa_socio_tem_negativacao,ifood_contagem_avaliacoes,ifood_faixa_preco,google_maps_avaliacao,google_maps_contagem_avaliacoes,google_maps_tem_website,inadimplente
0,5649308221613,PE,RECIFE,Restaurante Brasileiro,230-5 - Empresa Individual de Responsabilidade...,Fonte 5,56.11-2-01,Restaurantes e similares,150000.0,16500.0,0.0,0.0,NaN,0.0,350.0,$$$$$,4.25,1750.0,1.0,0.0
1,5728626180269,PE,RECIFE,Padaria,230-5 - Empresa Individual de Responsabilidade...,Fonte 5,10.91-1-02,Fabricação de produtos de padaria e confeitari...,85000.0,16500.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
2,5740206260397,PE,JABOATAO DOS GUARARAPES,Conveniência,206-2 - Sociedade Empresária Limitada,Fonte 5,46.13-3-00,Representantes comerciais e agentes do comérci...,9000.0,6500.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
3,5741028343981,PE,FERNANDO DE NORONHA,Hotel,206-2 - Sociedade Empresária Limitada,Fonte 5,55.10-8-01,Hotéis,9000.0,6500.0,0.0,0.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0
4,5741479985325,PE,RECIFE,Churrascaria,206-2 - Sociedade Empresária Limitada,Fonte 5,56.11-2-01,Restaurantes e similares,600000.0,16500.0,0.0,0.0,NaN,0.0,75.0,$,4.25,40.0,1.0,0.0


Como é possível observar no head acima, agora o dataset de clientes tem os dados em intervalos transformado em apenas um mesmo dado numérico, utizando a mediana do intervalo.

### 2.2 Valores Nulos

#### 2.2.1 Idenficação e Observação

Assim como observado na **Seção 2.4 da Fase 1**, todas as colunas com valores nulos carregas informação, vamos observar quais eram

In [34]:
# total de nulos e percentual em relação ao total de linhas
nulos = pd.DataFrame({
    'total_nulos': df_clientes_tratado.isnull().sum(),
    'pct_nulos': (df_clientes_tratado.isnull().sum() / len(df_clientes_tratado) * 100).round(1)
}).query('total_nulos > 0').sort_values('pct_nulos', ascending=False)

print('Colunas com valores nulos')
print(nulos.to_string())
print(f'\nTotal de colunas sem nenhum nulo: {df_clientes_tratado.shape[1] - len(nulos)} de {df_clientes_tratado.shape[1]}')

Colunas com valores nulos
                                 total_nulos  pct_nulos
serasa_credores                          171       85.5
ifood_contagem_avaliacoes                151       75.5
ifood_faixa_preco                        138       69.0
google_maps_avaliacao                    118       59.0
google_maps_contagem_avaliacoes          104       52.0
google_maps_tem_website                  104       52.0

Total de colunas sem nenhum nulo: 14 de 20


#### 2.2.2 Tratamento por Colunas de Valor Binário

Considerando essa realidade, a abordagem utilizada para este problema será **a criação de colunas binárias que identifiquem se há ou não valor naquele atributo**, exemplo: tem_ifood para identificar se o estabelecimento é cadastrado ou não no ifood.

In [35]:
# Colunas com nulos no dataset tratado
colunas_com_nulos = df_clientes_tratado.columns[df_clientes_tratado.isna().any()].tolist()
print('Colunas com nulos:')
print(colunas_com_nulos)

def criar_coluna_presenca(df, nome_coluna, colunas_base):
    colunas_existentes = [c for c in colunas_base if c in df.columns]
    if not colunas_existentes:
        print(f"Nao foi possivel criar {nome_coluna}: colunas base nao encontradas.")
        return
    
    # 1 se existe pelo menos um valor preenchido nas colunas base; 0 caso contrario
    df[nome_coluna] = df[colunas_existentes].notna().any(axis=1).astype(int)
    print(f"{nome_coluna} criada a partir de: {colunas_existentes}")

# Indicadores solicitados
criar_coluna_presenca(
    df_clientes_tratado,
    'tem_google_maps',
    [
        'google_maps_avaliacao',
        'google_maps_contagem_avaliacoes',
        'google_maps_tem_website'
    ]
)

criar_coluna_presenca(
    df_clientes_tratado,
    'tem_ifood',
    [
        'ifood_contagem_avaliacoes',
        'ifood_faixa_preco'
    ]
)

criar_coluna_presenca(
    df_clientes_tratado,
    'tem_credores',
    ['serasa_credores']
)

print('\nResumo das novas colunas binarias:')
for col in ['tem_google_maps', 'tem_ifood', 'tem_credores']:
    if col in df_clientes_tratado.columns:
        print(f"{col}:\n{df_clientes_tratado[col].value_counts(dropna=False).sort_index()}\n")

df_clientes_tratado[['tem_google_maps', 'tem_ifood', 'tem_credores']].head()

Colunas com nulos:
['serasa_credores', 'ifood_contagem_avaliacoes', 'ifood_faixa_preco', 'google_maps_avaliacao', 'google_maps_contagem_avaliacoes', 'google_maps_tem_website']
tem_google_maps criada a partir de: ['google_maps_avaliacao', 'google_maps_contagem_avaliacoes', 'google_maps_tem_website']
tem_ifood criada a partir de: ['ifood_contagem_avaliacoes', 'ifood_faixa_preco']
tem_credores criada a partir de: ['serasa_credores']

Resumo das novas colunas binarias:
tem_google_maps:
tem_google_maps
0    104
1     96
Name: count, dtype: int64

tem_ifood:
tem_ifood
0    138
1     62
Name: count, dtype: int64

tem_credores:
tem_credores
0    171
1     29
Name: count, dtype: int64



,tem_google_maps,tem_ifood,tem_credores
0,1,1,0
1,0,0,0
2,0,0,0
3,0,0,0
4,1,1,0


#### 2.2.3 Preenchimento de Valores Nulos

Agora que temos uma coluna para diferenciar os campos que são nulos e os que são realmente zero, vamos preencher o dataset de valores nulos com 0, pois considerando o contexto dos dados, é a estratégia mais eficiente.

In [36]:
# Preenchimento de nulos com 0 apos criacao das colunas de presenca
nulos_antes = int(df_clientes_tratado.isna().sum().sum())
print(f'Total de valores nulos antes: {nulos_antes}')

colunas_com_nulos_antes = df_clientes_tratado.columns[df_clientes_tratado.isna().any()].tolist()
print('Colunas com nulos antes do preenchimento:')
print(colunas_com_nulos_antes)

df_clientes_tratado = df_clientes_tratado.fillna(0)

nulos_depois = int(df_clientes_tratado.isna().sum().sum())
print(f'\nTotal de valores nulos depois: {nulos_depois}')
print('Preenchimento concluido.')

print('\nResumo rapido (primeiras linhas das colunas tratadas):')
print(df_clientes_tratado[colunas_com_nulos_antes].head())

Total de valores nulos antes: 786
Colunas com nulos antes do preenchimento:
['serasa_credores', 'ifood_contagem_avaliacoes', 'ifood_faixa_preco', 'google_maps_avaliacao', 'google_maps_contagem_avaliacoes', 'google_maps_tem_website']

Total de valores nulos depois: 0
Preenchimento concluido.

Resumo rapido (primeiras linhas das colunas tratadas):
  serasa_credores  ifood_contagem_avaliacoes ifood_faixa_preco  \
0               0                      350.0             $$$$$   
1               0                        0.0                 0   
2               0                        0.0                 0   
3               0                        0.0                 0   
4               0                       75.0                 $   

   google_maps_avaliacao  google_maps_contagem_avaliacoes  \
0                   4.25                           1750.0   
1                   0.00                              0.0   
2                   0.00                              0.0   
3          

In [37]:
df_clientes_tratado.head()

,id_cliente,uf,municipio,segmento_cliente,natureza_juridica,fonte_cliente,cnae_codigo,cnae_descricao,capital_social,idade_cnpj,...,serasa_socio_tem_negativacao,ifood_contagem_avaliacoes,ifood_faixa_preco,google_maps_avaliacao,google_maps_contagem_avaliacoes,google_maps_tem_website,inadimplente,tem_google_maps,tem_ifood,tem_credores
0,5649308221613,PE,RECIFE,Restaurante Brasileiro,230-5 - Empresa Individual de Responsabilidade...,Fonte 5,56.11-2-01,Restaurantes e similares,150000.0,16500.0,...,0.0,350.0,$$$$$,4.25,1750.0,1.0,0.0,1,1,0
1,5728626180269,PE,RECIFE,Padaria,230-5 - Empresa Individual de Responsabilidade...,Fonte 5,10.91-1-02,Fabricação de produtos de padaria e confeitari...,85000.0,16500.0,...,0.0,0.0,0,0.00,0.0,0.0,0.0,0,0,0
2,5740206260397,PE,JABOATAO DOS GUARARAPES,Conveniência,206-2 - Sociedade Empresária Limitada,Fonte 5,46.13-3-00,Representantes comerciais e agentes do comérci...,9000.0,6500.0,...,0.0,0.0,0,0.00,0.0,0.0,0.0,0,0,0
3,5741028343981,PE,FERNANDO DE NORONHA,Hotel,206-2 - Sociedade Empresária Limitada,Fonte 5,55.10-8-01,Hotéis,9000.0,6500.0,...,0.0,0.0,0,0.00,0.0,0.0,0.0,0,0,0
4,5741479985325,PE,RECIFE,Churrascaria,206-2 - Sociedade Empresária Limitada,Fonte 5,56.11-2-01,Restaurantes e similares,600000.0,16500.0,...,0.0,75.0,$,4.25,40.0,1.0,0.0,1,1,0


### 2.3 Transformação de Variavéis Categóricas

Nesta subseção, as variavéis categóricas serão tratadas para se adequarem a colunas de valor numérico

## 3. Tratamento de Dados Complexos

Nesta seção será tratado colunas com valores complexos, como listas e códigos


## 4. Agregação e Criação de Colunas

Nesta seção há o processo de agregação e criação de colunas do dataset de Clientes a partir dos insights sobre possíveis features importantes

## 5. Salvando o Novo Dataset

Nesta seção será salvo o novo dataset, além de explicitar a interpretação por cada nova coluna adicionada

### 5.1 Salvamento Provisório

**ATUALIZAR DEPOIS**: Salvamento provisório de um dataset tratado com resolução de intervaloes e valores nulos

In [42]:
# Pasta e nome do arquivo de saida
pasta_saida = '../dados/tratados'
nome_arquivo = 'clientes_filtrado_tratado2_incompleto.csv'
os.makedirs(pasta_saida, exist_ok=True)

# Usa df_clientes_tratado para salvar um csv
if 'df_clientes_tratado' in globals():
    df_para_salvar = df_clientes_tratado.copy()
    nome_origem = 'clientes_filtrado'


caminho_saida = os.path.join(pasta_saida, nome_arquivo)
df_para_salvar.to_csv(caminho_saida, index=False, encoding='utf-8-sig')

print(f'Dataset salvo com sucesso.')
print(f'Origem: {nome_origem}')
print(f'Caminho: {os.path.abspath(caminho_saida)}')
print(f'Dimensao: {df_para_salvar.shape[0]} linhas x {df_para_salvar.shape[1]} colunas')

Dataset salvo com sucesso.
Origem: clientes_filtrado
Caminho: c:\Users\Fellype Fontes\Documents\UFPB\Aprendizagem de Maquina\PrevisaoInadimplecia-ML\dados\tratados\clientes_filtrado_tratado2_incompleto.csv
Dimensao: 200 linhas x 23 colunas


In [44]:
df_clientes_tratado.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id_cliente                       200 non-null    int64  
 1   uf                               200 non-null    str    
 2   municipio                        200 non-null    str    
 3   segmento_cliente                 200 non-null    str    
 4   natureza_juridica                200 non-null    str    
 5   fonte_cliente                    200 non-null    str    
 6   cnae_codigo                      200 non-null    str    
 7   cnae_descricao                   200 non-null    str    
 8   capital_social                   200 non-null    float64
 9   idade_cnpj                       200 non-null    float64
 10  serasa_contagem_negativacoes     200 non-null    float64
 11  serasa_contagem_protestos        200 non-null    float64
 12  serasa_credores                  

## 6. Resumo Geral das Ativadades